In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv('/content/LC_loans_granting_model_dataset.csv')

/tmp/ipykernel_1766/3624295061.py:1: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/LC_loans_granting_model_dataset.csv')


In [5]:
pd.set_option("display.max_columns", None)

In [6]:
df.head()

,id,issue_d,revenue,dti_n,loan_amnt,fico_n,experience_c,emp_length,purpose,home_ownership_n,addr_state,zip_code,Default,title,desc
0,68407277,Dec-2015,55000.0,5.91,3600,677.0,1,10+ years,debt_consolidation,MORTGAGE,PA,190xx,0.0,Debt consolidation,NaN
1,68355089,Dec-2015,65000.0,16.06,24700,717.0,1,10+ years,small_business,MORTGAGE,SD,577xx,0.0,Business,NaN
2,68341763,Dec-2015,71000.0,13.85,20000,697.0,1,10+ years,home_improvement,MORTGAGE,IL,605xx,0.0,NaN,NaN
3,68476807,Dec-2015,104433.0,25.37,10400,697.0,1,3 years,major_purchase,MORTGAGE,PA,174xx,0.0,Major purchase,NaN
4,68426831,Dec-2015,34000.0,10.20,11950,692.0,1,4 years,debt_consolidation,RENT,GA,300xx,0.0,Debt consolidation,NaN


In [7]:
df.shape

(1242821, 15)

In [9]:
df.columns.tolist()

['id',
 'issue_d',
 'revenue',
 'dti_n',
 'loan_amnt',
 'fico_n',
 'experience_c',
 'emp_length',
 'purpose',
 'home_ownership_n',
 'addr_state',
 'zip_code',
 'Default',
 'title',
 'desc']

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1242821 entries, 0 to 1242820
Data columns (total 15 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   id                1242821 non-null  int64  
 1   issue_d           1242821 non-null  object 
 2   revenue           1242821 non-null  float64
 3   dti_n             1242821 non-null  float64
 4   loan_amnt         1242821 non-null  int64  
 5   fico_n            1242821 non-null  float64
 6   experience_c      1242821 non-null  int64  
 7   emp_length        1242821 non-null  object 
 8   purpose           1242820 non-null  object 
 9   home_ownership_n  1242820 non-null  object 
 10  addr_state        1242820 non-null  object 
 11  zip_code          1242819 non-null  object 
 12  Default           1242820 non-null  float64
 13  title             1226695 non-null  object 
 14  desc              119098 non-null   object 
dtypes: float64(4), int64(3), object(8)
memory usage: 

In [11]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
id,1242821.0,5.238381e+07,3.728352e+07,54734.0,16271176.00,53392291.00,75210302.00,145636374.0
revenue,1242821.0,7.693937e+04,6.917134e+04,1896.0,46000.00,65000.00,92000.00,10999200.0
dti_n,1242821.0,1.825537e+01,1.067032e+01,0.0,11.81,17.61,24.03,999.0
loan_amnt,1242821.0,1.442894e+04,8.665202e+03,500.0,8000.00,12000.00,20000.00,40000.0
fico_n,1242821.0,6.979569e+02,3.167068e+01,612.0,672.00,692.00,712.00,847.5
experience_c,1242821.0,9.999799e-01,4.484990e-03,0.0,1.00,1.00,1.00,1.0
Default,1242820.0,1.969271e-01,3.976770e-01,0.0,0.00,0.00,0.00,1.0


In [13]:
#missing values
missing = df.isnull().sum()

missing = (
    missing[missing > 0]
    .sort_values(ascending=False)
)

missing

,0
desc,1123723
title,16126
zip_code,2
addr_state,1
home_ownership_n,1
purpose,1
Default,1


In [14]:
missing_pct = (
    df.isnull().mean() * 100
).sort_values(ascending=False)

missing_pct.head(20)

,0
desc,90.417124
title,1.297532
zip_code,0.000161
purpose,0.000080
Default,0.000080
addr_state,0.000080
home_ownership_n,0.000080
experience_c,0.000000
fico_n,0.000000
loan_amnt,0.000000


In [16]:
#duplicates
df.duplicated().sum()
df[df.duplicated()]

,id,issue_d,revenue,dti_n,loan_amnt,fico_n,experience_c,emp_length,purpose,home_ownership_n,addr_state,zip_code,Default,title,desc


In [19]:
df_clean = df.copy()
df_clean = df_clean.drop_duplicates().copy()

print("Shape after removing duplicates:", df_clean.shape)

Shape after removing duplicates: (1242821, 15)


In [20]:
print("Number of columns:", len(df_clean.columns))

for i, col in enumerate(df_clean.columns, start=1):
    print(i, ":", col)

Number of columns: 15
1 : id
2 : issue_d
3 : revenue
4 : dti_n
5 : loan_amnt
6 : fico_n
7 : experience_c
8 : emp_length
9 : purpose
10 : home_ownership_n
11 : addr_state
12 : zip_code
13 : Default
14 : title
15 : desc


In [21]:
status_columns = [
    col for col in df_clean.columns
    if any(word in col.lower()
           for word in ["status", "default", "loan_status"])
]

print("Possible target/status columns:")
print(status_columns)

Possible target/status columns:
['Default']


In [23]:
df_clean["Default"].value_counts(dropna=False)

,count
Default,
0.0,998075
1.0,244745
NaN,1


In [24]:
df_clean["Default"].value_counts(normalize=True, dropna=False) * 100

,proportion
Default,
0.0,80.307220
1.0,19.692699
NaN,0.000080


In [25]:
print("Missing Default values:",
      df_clean["Default"].isna().sum())

Missing Default values: 1


In [26]:
df_clean = df_clean.dropna(
    subset=["Default"]
).copy()

In [27]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1242820 entries, 0 to 1242819
Data columns (total 15 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   id                1242820 non-null  int64  
 1   issue_d           1242820 non-null  object 
 2   revenue           1242820 non-null  float64
 3   dti_n             1242820 non-null  float64
 4   loan_amnt         1242820 non-null  int64  
 5   fico_n            1242820 non-null  float64
 6   experience_c      1242820 non-null  int64  
 7   emp_length        1242820 non-null  object 
 8   purpose           1242820 non-null  object 
 9   home_ownership_n  1242820 non-null  object 
 10  addr_state        1242820 non-null  object 
 11  zip_code          1242819 non-null  object 
 12  Default           1242820 non-null  float64
 13  title             1226695 non-null  object 
 14  desc              119098 non-null   object 
dtypes: float64(4), int64(3), object(8)
memory usage: 151.7

In [28]:
df_clean.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,1242820.0,NaN,NaN,NaN,52383786.453269,37283519.342177,54734.0,16271171.25,53392275.0,75210301.25,145636374.0
issue_d,1242820,133,Mar-2016,48927,NaN,NaN,NaN,NaN,NaN,NaN,NaN
revenue,1242820.0,NaN,NaN,NaN,76939.401748,69171.360564,1896.0,46000.0,65000.0,92000.0,10999200.0
dti_n,1242820.0,NaN,NaN,NaN,18.255363,10.670318,0.0,11.81,17.61,24.03,999.0
loan_amnt,1242820.0,NaN,NaN,NaN,14428.945583,8665.201264,500.0,8000.0,12000.0,20000.0,40000.0
fico_n,1242820.0,NaN,NaN,NaN,697.95689,31.670696,612.0,672.0,692.0,712.0,847.5
experience_c,1242820.0,NaN,NaN,NaN,0.99998,0.004485,0.0,1.0,1.0,1.0,1.0
emp_length,1242820,12,10+ years,407594,NaN,NaN,NaN,NaN,NaN,NaN,NaN
purpose,1242820,14,debt_consolidation,721292,NaN,NaN,NaN,NaN,NaN,NaN,NaN
home_ownership_n,1242820,4,MORTGAGE,614697,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
#for unique values
for col in df_clean.columns:
    print("\n" + "="*50)
    print(col)
    print("Unique values:", df_clean[col].nunique())
    print(df_clean[col].head(10).tolist())


id
Unique values: 1242820
[68407277, 68355089, 68341763, 68476807, 68426831, 68476668, 67275481, 68466926, 68616873, 68338832]

issue_d
Unique values: 133
['Dec-2015', 'Dec-2015', 'Dec-2015', 'Dec-2015', 'Dec-2015', 'Dec-2015', 'Dec-2015', 'Dec-2015', 'Dec-2015', 'Dec-2015']

revenue
Unique values: 62382
[55000.0, 65000.0, 71000.0, 104433.0, 34000.0, 180000.0, 85000.0, 85000.0, 42000.0, 64000.0]

dti_n
Unique values: 6635
[5.91, 16.06, 13.85, 25.37, 10.2, 14.67, 17.61, 13.07, 34.8, 34.95]

loan_amnt
Unique values: 1549
[3600, 24700, 20000, 10400, 11950, 20000, 20000, 10000, 8000, 1400]

fico_n
Unique values: 48
[677.0, 717.0, 697.0, 697.0, 692.0, 682.0, 707.0, 687.0, 702.0, 702.0]

experience_c
Unique values: 2
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

emp_length
Unique values: 12
['10+ years', '10+ years', '10+ years', '3 years', '4 years', '10+ years', '10+ years', '6 years', '10+ years', '3 years']

purpose
Unique values: 14
['debt_consolidation', 'small_business', 'home_improvement', 'major

In [30]:
columns_to_remove = [
    "id",
    "zip_code",
    "title",
    "desc"
]

df_clean = df_clean.drop(
    columns=columns_to_remove,
    errors="ignore"
)

print(df_clean.columns.tolist())

['issue_d', 'revenue', 'dti_n', 'loan_amnt', 'fico_n', 'experience_c', 'emp_length', 'purpose', 'home_ownership_n', 'addr_state', 'Default']


In [31]:
missing_summary = pd.DataFrame({
    "Missing Count": df_clean.isna().sum(),
    "Missing %": df_clean.isna().mean() * 100
})

missing_summary.sort_values(
    "Missing %",
    ascending=False
)

,Missing Count,Missing %
issue_d,0,0.0
revenue,0,0.0
dti_n,0,0.0
loan_amnt,0,0.0
fico_n,0,0.0
experience_c,0,0.0
emp_length,0,0.0
purpose,0,0.0
home_ownership_n,0,0.0
addr_state,0,0.0


In [36]:
numeric_columns = [
    "revenue",
    "dti_n",
    "loan_amnt",
    "fico_n",
    "experience_c"
]

numeric_columns = [
    col for col in numeric_columns
    if col in df_clean.columns
]

df_clean[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
revenue,1242820.0,76939.401748,69171.360564,1896.0,46000.00,65000.00,92000.00,10999200.0
dti_n,1242820.0,18.255363,10.670318,0.0,11.81,17.61,24.03,999.0
loan_amnt,1242820.0,14428.945583,8665.201264,500.0,8000.00,12000.00,20000.00,40000.0
fico_n,1242820.0,697.956890,31.670696,612.0,672.00,692.00,712.00,847.5
experience_c,1242820.0,0.999980,0.004485,0.0,1.00,1.00,1.00,1.0


In [37]:
for col in numeric_columns:
    print(
        col,
        "negative values:",
        (df_clean[col]< 0).sum()
    )

revenue negative values: 0
dti_n negative values: 0
loan_amnt negative values: 0
fico_n negative values: 0
experience_c negative values: 0


In [38]:
print("Zero loan amounts:",
      (df_clean["loan_amnt"] == 0).sum())

print("Zero revenue:",
      (df_clean["revenue"] == 0).sum())

print("Negative DTI:",
      (df_clean["dti_n"] < 0).sum())

print("Negative FICO:",
      (df_clean["fico_n"] < 0).sum())

Zero loan amounts: 0
Zero revenue: 0
Negative DTI: 0
Negative FICO: 0


In [39]:
categorical_columns = [
    "purpose",
    "home_ownership_n",
    "addr_state"
]

for col in categorical_columns:
    if col in df_clean.columns:
        print("\n", "="*50)
        print(col)
        print(df_clean[col].value_counts(dropna=False).head(20))


purpose
purpose
debt_consolidation    721292
credit_card           275931
home_improvement       80001
other                  69943
major_purchase         26893
small_business         14411
medical                14000
car                    13453
moving                  8674
vacation                8182
house                   6425
wedding                 2350
renewable_energy         842
educational              423
Name: count, dtype: int64

home_ownership_n
home_ownership_n
MORTGAGE    614697
RENT        495509
OWN         132156
OTHER          458
Name: count, dtype: int64

addr_state
addr_state
CA    181911
NY    102054
TX    101394
FL     87842
IL     48049
NJ     45082
PA     42223
OH     40600
GA     40095
VA     35495
NC     34799
MI     32427
AZ     29886
MD     28908
MA     28739
CO     27244
WA     27086
MN     22185
IN     19843
MO     19656
Name: count, dtype: int64


In [40]:
df_clean["issue_d"].head(10)

,issue_d
0,Dec-2015
1,Dec-2015
2,Dec-2015
3,Dec-2015
4,Dec-2015
5,Dec-2015
6,Dec-2015
7,Dec-2015
8,Dec-2015
9,Dec-2015


In [41]:
df_clean["issue_d"].dtype

dtype('O')

In [42]:
df_clean["issue_d"] = pd.to_datetime(
    df_clean["issue_d"],
    errors="coerce"
)

/tmp/ipykernel_1766/3148582013.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean["issue_d"] = pd.to_datetime(


In [43]:
df_clean["issue_d"].describe()

,issue_d
count,1242820
mean,2015-04-10 14:37:27.744966912
min,2007-06-01 00:00:00
25%,2014-05-01 00:00:00
50%,2015-07-01 00:00:00
75%,2016-03-01 00:00:00
max,2018-12-01 00:00:00


In [44]:
df_clean["issue_year"] = (
    df_clean["issue_d"].dt.year
)

df_clean["issue_month"] = (
    df_clean["issue_d"].dt.month
)

In [45]:
df_clean["issue_quarter"] = (
    df_clean["issue_d"].dt.quarter
)

In [46]:
df_clean["loan_to_revenue"] = (
    df_clean["loan_amnt"] /
    df_clean["revenue"]
)

In [47]:
df_clean["fico_band"] = pd.cut(
    df_clean["fico_n"],
    bins=[0, 580, 670, 740, 800, 900],
    labels=[
        "Poor",
        "Fair",
        "Good",
        "Very Good",
        "Excellent"
    ]
)

In [48]:
df_clean["dti_band"] = pd.cut(
    df_clean["dti_n"],
    bins=[-np.inf, 10, 20, 30, 40, np.inf],
    labels=[
        "Very Low",
        "Low",
        "Moderate",
        "High",
        "Very High"
    ]
)

In [49]:
df_clean[
    [
        "revenue",
        "loan_amnt",
        "loan_to_revenue",
        "fico_n",
        "fico_band",
        "dti_n",
        "dti_band",
        "Default"
    ]
].head(20)

,revenue,loan_amnt,loan_to_revenue,fico_n,fico_band,dti_n,dti_band,Default
0,55000.0,3600,0.065455,677.0,Good,5.91,Very Low,0.0
1,65000.0,24700,0.380000,717.0,Good,16.06,Low,0.0
2,71000.0,20000,0.281690,697.0,Good,13.85,Low,0.0
3,104433.0,10400,0.099585,697.0,Good,25.37,Moderate,0.0
4,34000.0,11950,0.351471,692.0,Good,10.20,Low,0.0
5,180000.0,20000,0.111111,682.0,Good,14.67,Low,0.0
6,85000.0,20000,0.235294,707.0,Good,17.61,Low,0.0
7,85000.0,10000,0.117647,687.0,Good,13.07,Low,0.0
8,42000.0,8000,0.190476,702.0,Good,34.80,High,0.0
9,64000.0,1400,0.021875,702.0,Good,34.95,High,0.0


In [50]:
df_clean = df_clean.replace(
    [np.inf, -np.inf],
    np.nan
)

In [51]:
df_clean.isna().sum().sort_values(
    ascending=False
)

,0
issue_d,0
revenue,0
dti_n,0
loan_amnt,0
fico_n,0
experience_c,0
emp_length,0
purpose,0
home_ownership_n,0
addr_state,0


In [52]:
for col in numeric_columns:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(
            df_clean[col].median()
        )

In [53]:
for col in categorical_columns:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(
            "Unknown"
        )

In [54]:
df_clean.isna().sum().sort_values(
    ascending=False
)

,0
issue_d,0
revenue,0
dti_n,0
loan_amnt,0
fico_n,0
experience_c,0
emp_length,0
purpose,0
home_ownership_n,0
addr_state,0


In [55]:
print(
    "Duplicates:",
    df_clean.duplicated().sum()
)

Duplicates: 0


In [56]:
print("Rows:", df_clean.shape[0])
print("Columns:", df_clean.shape[1])

print("\nColumns:")
print(df_clean.columns.tolist())

print("\nDefault distribution:")
print(df_clean["Default"].value_counts())

print("\nDefault percentage:")
print(
    df_clean["Default"]
    .value_counts(normalize=True) * 100
)

Rows: 1242820
Columns: 17

Columns:
['issue_d', 'revenue', 'dti_n', 'loan_amnt', 'fico_n', 'experience_c', 'emp_length', 'purpose', 'home_ownership_n', 'addr_state', 'Default', 'issue_year', 'issue_month', 'issue_quarter', 'loan_to_revenue', 'fico_band', 'dti_band']

Default distribution:
Default
0.0    998075
1.0    244745
Name: count, dtype: int64

Default percentage:
Default
0.0    80.307285
1.0    19.692715
Name: proportion, dtype: float64


In [58]:
import os

processed_path = (
    "/content/drive/MyDrive/"
    "FinTech Credit Risk Project/"
    "Data/Processed/"
    "cleaned_credit_risk_data.csv"
)

# Extract the directory path
output_dir = os.path.dirname(processed_path)

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

df_clean.to_csv(
    processed_path,
    index=False
)

print("Saved to:")
print(processed_path)

Saved to:
/content/drive/MyDrive/FinTech Credit Risk Project/Data/Processed/cleaned_credit_risk_data.csv


In [60]:
df_clean["Default"].value_counts(dropna=False)

,count
Default,
0.0,998075
1.0,244745


In [61]:
df_clean[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
revenue,1242820.0,76939.401748,69171.360564,1896.0,46000.00,65000.00,92000.00,10999200.0
dti_n,1242820.0,18.255363,10.670318,0.0,11.81,17.61,24.03,999.0
loan_amnt,1242820.0,14428.945583,8665.201264,500.0,8000.00,12000.00,20000.00,40000.0
fico_n,1242820.0,697.956890,31.670696,612.0,672.00,692.00,712.00,847.5
experience_c,1242820.0,0.999980,0.004485,0.0,1.00,1.00,1.00,1.0


In [59]:
for col in ["purpose", "home_ownership_n", "experience_c", "emp_length"]:
    print("\n", col)
    print(df_clean[col].value_counts(dropna=False).head(20))


 purpose
purpose
debt_consolidation    721292
credit_card           275931
home_improvement       80001
other                  69943
major_purchase         26893
small_business         14411
medical                14000
car                    13453
moving                  8674
vacation                8182
house                   6425
wedding                 2350
renewable_energy         842
educational              423
Name: count, dtype: int64

 home_ownership_n
home_ownership_n
MORTGAGE    614697
RENT        495509
OWN         132156
OTHER          458
Name: count, dtype: int64

 experience_c
experience_c
1    1242795
0         25
Name: count, dtype: int64

 emp_length
emp_length
10+ years    407594
2 years      112343
< 1 year      99567
3 years       99423
1 year        81839
5 years       77864
4 years       74485
NI            70988
6 years       58149
8 years       57108
7 years       56223
9 years       47237
Name: count, dtype: int64


In [62]:
df_clean["issue_d"] = pd.to_datetime(
    df_clean["issue_d"],
    format="%b-%Y",
    errors="coerce"
)

In [63]:
print("Missing issue dates:", df_clean["issue_d"].isna().sum())
print(df_clean["issue_d"].min())
print(df_clean["issue_d"].max())

Missing issue dates: 0
2007-06-01 00:00:00
2018-12-01 00:00:00
